# Group 8 — ISIC skin-lesion classification

Coding component skeleton. Every requirement from the client document maps to one section below.
Target runtime: local conda dev environment (Anaconda + Jupyter, Python 3.12, TensorFlow 2.21),
single TensorFlow / Keras stack.
Fill in the TODO blocks in order; each section must stay runnable from top to bottom.

**Datasets (both are used)**
- ISIC 2018 Task 3 (HAM10000, 7 classes) — train / validation / internal test, lesion-disjoint split
- ISIC 2019 (8 classes) — external validation, SCC excluded, hash-deduped against 2018

The third dataset listed in the client document (SIIM-ISIC 2020) is not used: its archive is over 100 GB.
This deviation is recorded in `AGENTS.md` and must be stated in the report.

**Fixed label space (7 classes)**: MEL, NV, BCC, AKIEC, BKL, DF, VASC


## Requirement map

| Section | Requirement | Main artifact |
|---|---|---|
| 0 | setup | paths, output folders, dataset file counts |
| 1 | 2.1 | library version table |
| 2 | 2.2 | gradient tape demonstration |
| 3 | 2.3 | training vs evaluation memory |
| data | foundation | `isic2018` / `isic2019` tables |
| 5 | 2.4 | preprocessing and label encoding |
| 6 | 2.5–2.6 | augmentation figure |
| 7 | 2.7–2.9 | `model.summary()`, feature maps |
| 8 | 1.1 | gradient-magnitude figure, VGG vs ResNet |
| 9 | 2.10–2.13 | training history and learning curves |
| 10 | 2.14–2.17 | confusion matrix, top-3 confident errors |
| 11 | 2.18–2.20 | external results, unseen image, calibration |
| 12 | submission | saved artifacts and results archive |

Coverage checklist: 1.1 · 2.1 · 2.2 · 2.3 · 2.4 · 2.5 · 2.6 · 2.7 · 2.8 · 2.9 · 2.10 · 2.11 ·
2.12 · 2.13 · 2.14 · 2.15 · 2.16 · 2.17 · 2.18 · 2.19 · 2.20


## Run plan

- **Once, before running anything**: download the two Kaggle datasets and unpack them into `data/`
  (the exact folder names are listed in section 0).
- **Session 1**: sections 0 → data layer; the printed counts must be larger than zero.
- **Session 2**: sections 2.1 → 2.9.
- **Session 3**: sections 2.10 → 2.13 (training, checkpoint each epoch).
- **Session 4**: sections 2.14 → 2.20 (evaluation and figures).


## 0. Setup and runtime

Download the two datasets from Kaggle **once** and unpack them into this project like this:

```
data/
  ISIC 2018/   HAM10000_metadata.csv + *.jpg   (HAM10000, 7 classes, 10,015 images)
  ISIC 2019/   ISIC_2019_Training_GroundTruth.csv + *.jpg
```

`data/` and `outputs/` are git-ignored, so the images never enter the repository.
The last line of the cell prints how many JPEGs are visible per dataset; a wrong folder name shows up
immediately as `0 jpg`.

Produces: fixed paths, output folders, seeds, dataset file counts.


In [ ]:
# Section 0 - paths and seeds. The two datasets are downloaded once from Kaggle and unpacked here:
#   data/ISIC 2018/  data/ISIC 2019/
import os, random, hashlib
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # keep TensorFlow C++ logs out of the notebook output
from pathlib import Path
import numpy as np
import pandas as pd

def project_root(start: Path) -> Path:
    """Return the project root whether the kernel starts in the root or in code/ (local Jupyter)."""
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() or (candidate / "AGENTS.md").is_file():
            return candidate
    raise RuntimeError("open this notebook inside the project folder (data/ not found)")

ROOT = project_root(Path.cwd())
os.chdir(ROOT)                # keep every relative path below stable and portable
DATA_ROOT = Path("data")      # project folder; keep the same two subfolders
OUT_DIR = Path("outputs")     # figures and tables produced by later sections
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]  # fixed 7-class label space
IMG_SIZE = 224
BATCH_SIZE = 16

SEED = 8
random.seed(SEED)
np.random.seed(SEED)

assert DATA_ROOT.is_dir(), "create data/ with the two dataset folders first"
# First sanity check: every line must show a number larger than 0.
for name in ("ISIC 2018", "ISIC 2019"):
    print(f"{name:10s} {len(list((DATA_ROOT / name).rglob('*.jpg'))):6d} jpg")


## 2.1 Libraries

Required: list every library used and its version. Produces: printed version table saved to `env.json`.

In [ ]:
# 2.1 - Record every library used and its version (saved to outputs/tables/env.json).
import warnings
warnings.filterwarnings("ignore")
import json, math, platform, time
import tensorflow as tf
import keras
import sklearn
import PIL
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# TF 2.21 / CPU: the MLIR meta optimizer prints spurious "operand does not dominate" errors for
# GradientTape graphs. Disabling it keeps the output clean and leaves training speed unchanged.
tf.config.optimizer.set_experimental_options({"disable_meta_optimizer": True})

ENV = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": PIL.__version__,
}
env_df = pd.DataFrame({"library": list(ENV), "version": list(ENV.values())})
print(env_df.to_string(index=False))
(OUT_DIR / "tables" / "env.json").write_text(json.dumps(ENV, indent=2))

# One seed for python, numpy and tensorflow; section 0 already fixed SEED.
tf.keras.utils.set_random_seed(SEED)
print("\nrandom seed:", SEED)


## 2.2 Forward pass and automatic differentiation

Required: explain how the framework tracks the forward pass and recovers gradients. Produces: per-layer gradient norms for one real batch.

**What the tape records.** While the `GradientTape` context is open, TensorFlow stores every operation whose inputs include a trainable variable together with the tensors needed to differentiate it. Closing the context freezes that recording; `tape.gradient()` then replays it in reverse (chain rule) and returns one gradient tensor per trainable variable. The tape keeps references to intermediate activations until it is consumed, which is why training needs more memory than inference.


In [ ]:
# 2.2 - A real image batch, one real forward pass, and the gradients the tape recovers.
def load_raw_batch(folder, n=8):
    """Read JPEGs straight from disk: this is the autodiff demo, full preprocessing comes in 2.4."""
    paths = sorted(p for p in (DATA_ROOT / folder).rglob("*.jpg"))[:n]
    batch = [np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
             for p in paths]
    return tf.convert_to_tensor(np.stack(batch)), paths

demo_x, demo_paths = load_raw_batch("ISIC 2018")
demo_y = tf.one_hot([0, 1, 2, 3, 4, 5, 6, 0], depth=len(CLASS_NAMES))

demo_model = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.Conv2D(32, 3, strides=2, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="tape_demo")
demo_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# Every operation that depends on a trainable variable is recorded while the context is open.
with tf.GradientTape() as tape:
    logits = demo_model(demo_x, training=True)
    loss = demo_loss(demo_y, logits)

# gradient() replays the recorded graph backwards (chain rule) and returns one tensor per variable.
grads = tape.gradient(loss, demo_model.trainable_variables)
print("demo batch:", tuple(demo_x.shape), "| loss:", round(float(loss), 4))

grad_rows = [{"layer": var.name, "shape": tuple(var.shape),
              "grad_norm": float(tf.norm(g)) if g is not None else 0.0}
             for var, g in zip(demo_model.trainable_variables, grads)]
grad_df = pd.DataFrame(grad_rows)
print(grad_df.to_string(index=False))
grad_df.to_csv(OUT_DIR / "tables" / "gradient_norms_2_2.csv", index=False)


## 2.3 Training and evaluation memory

Required: explain the memory difference between training and evaluation mode. Produces: behaviour evidence and written comparison.

**Why training costs more memory.** In training mode the framework keeps every intermediate activation needed by the backward pass, BatchNormalization uses the statistics of the current batch, and dropout samples a new mask on every call. In evaluation mode no backward pass follows, so activations can be released as soon as they are produced, BatchNormalization reuses the moving statistics learned during training, and dropout is switched off - the output is deterministic and no tape is needed. The printed comparison shows the two regimes: training-mode calls disagree with each other, evaluation-mode calls are identical.


In [ ]:
# 2.3 - Training mode versus evaluation mode: behaviour evidence and cost evidence.
demo_model3 = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="mode_demo")

train_out_a = demo_model3(demo_x, training=True)
train_out_b = demo_model3(demo_x, training=True)
eval_out_a = demo_model3(demo_x, training=False)
eval_out_b = demo_model3(demo_x, training=False)
print("training=True  -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(train_out_a - train_out_b))), 6))
print("training=False -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(eval_out_a - eval_out_b))), 6))

@tf.function
def forward_eval(x):
    return demo_model3(x, training=False)

@tf.function
def forward_train_with_tape(x):
    with tf.GradientTape() as tape:
        y = demo_model3(x, training=True)
        loss = tf.reduce_mean(y)
    return tape.gradient(loss, demo_model3.trainable_variables)

def time_forward(fn, repeats=3):
    fn(demo_x)                      # first call traces the tf.function
    start = time.perf_counter()
    for _ in range(repeats):
        fn(demo_x)
    return (time.perf_counter() - start) / repeats

t_eval = time_forward(forward_eval)
t_train = time_forward(forward_train_with_tape)
print(f"eval forward: {t_eval * 1000:.1f} ms | training forward + backward: {t_train * 1000:.1f} ms")
print(f"one training step costs {t_train / t_eval:.1f}x the evaluation pass on this batch")


## Data layer: load the two datasets from the project folder

No download code is needed: the folders above are read directly. Each dataset becomes one small table
with the columns the later sections need (`image`, `path`, `label` / `target`, `split`, `md5`).

Rules applied here:
- fixed 7-class label space; HAM10000 lowercase codes are mapped to class names;
- lesion-level split for ISIC 2018, so all images of one lesion stay in one split;
- SCC dropped from ISIC 2019 (the model has no SCC slot);
- duplicate images removed from ISIC 2019 by file md5 (it largely repeats ISIC 2018).

Produces: the `isic2018` and `isic2019` tables, plus `outputs/tables/isic2018_split.csv`.


In [ ]:
# 2.4 - ISIC 2018 (HAM10000): build image path + 7-class label + lesion id.
# HAM10000 stores lowercase codes such as "mel"; map them to the fixed class names.
DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AKIEC",
               "bkl": "BKL", "df": "DF", "vasc": "VASC"}

base = DATA_ROOT / "ISIC 2018"
meta_path = next(base.rglob("HAM10000_metadata.csv"), None)
assert meta_path is not None, "data/ISIC 2018 must contain HAM10000_metadata.csv"
meta = pd.read_csv(meta_path)
jpgs = {p.stem: p for p in base.rglob("*.jpg")}     # image name -> full path

isic2018 = meta[["image_id", "lesion_id", "dx"]].rename(columns={"image_id": "image"})
isic2018["label"] = isic2018["dx"].map(DX_TO_CLASS)
isic2018["path"] = isic2018["image"].map(lambda name: str(jpgs[name]))
print(isic2018["label"].value_counts().to_string())


In [ ]:
# Lesion-level split: all images of one lesion land in the same split (no leakage between splits).
lesions = sorted(isic2018["lesion_id"].unique())
np.random.RandomState(SEED).shuffle(lesions)
val_lesions = set(lesions[int(len(lesions) * 0.70):int(len(lesions) * 0.85)])
test_lesions = set(lesions[int(len(lesions) * 0.85):])

def split_of(lesion_id):
    if lesion_id in val_lesions:
        return "validation"
    if lesion_id in test_lesions:
        return "test"
    return "train"

isic2018["split"] = isic2018["lesion_id"].map(split_of)
isic2018.to_csv(OUT_DIR / "tables" / "isic2018_split.csv", index=False)
print(isic2018["split"].value_counts().to_string())


In [ ]:
# 2.4 - ISIC 2019 is the external validation set. It has 8 classes; SCC has no slot in our 7 classes.
# The ground-truth CSV stores AKIEC as "AK", so the column names are mapped to the fixed class names.
ISIC2019_CLASSES = {"MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AKIEC",
                    "BKL": "BKL", "DF": "DF", "VASC": "VASC", "SCC": "SCC"}

base = DATA_ROOT / "ISIC 2019"
jpgs = {p.stem: p for p in base.rglob("*.jpg")}
truth_path = next(base.rglob("ISIC_2019_Training_GroundTruth.csv"), None)
assert truth_path is not None, "data/ISIC 2019 must contain ISIC_2019_Training_GroundTruth.csv"
truth = pd.read_csv(truth_path)
truth["label"] = truth[list(ISIC2019_CLASSES)].idxmax(axis=1).map(ISIC2019_CLASSES)
scc_count = int((truth["label"] == "SCC").sum())
isic2019 = truth[truth["label"] != "SCC"][["image", "label"]].copy()
isic2019["path"] = isic2019["image"].map(lambda name: str(jpgs[name]))
isic2019["split"] = "external_val"
print("isic2019:", len(isic2019), "| SCC dropped:", scc_count)


In [ ]:
# ISIC 2019 largely repeats ISIC 2018 images, so remove them: the external set must be unseen.
def md5(path):
    with open(path, "rb") as handle:
        return hashlib.md5(handle.read()).hexdigest()

isic2018["md5"] = [md5(p) for p in isic2018["path"]]
isic2019["md5"] = [md5(p) for p in isic2019["path"]]
duplicates = isic2019["md5"].isin(isic2018["md5"])
isic2019 = isic2019[~duplicates].reset_index(drop=True)
print("duplicate images removed from isic2019:", int(duplicates.sum()))
print("rows -> isic2018:", len(isic2018), "| isic2019:", len(isic2019))


## 2.4 Preprocessing and label encoding

Required: preprocessing, augmentation and label encoding. Produces: fixed class order, uint8 cache, `tf.data` pipeline, one-hot labels.

**Preprocessing here.** Each JPEG is decoded and resized to `IMG_SIZE` once; the `uint8` result is cached under `outputs/cache/` so later sessions skip JPEG decoding. Labels use the fixed `CLASS_NAMES` order and are one-hot encoded; the three `tf.data` pipelines shuffle (train only), batch and prefetch.


In [ ]:
# 2.4 - Preprocessing and label encoding: decode/resize once, cache uint8 arrays, build tf.data pipelines.
CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

def decode_resize(path):
    """JPEG bytes -> uint8 RGB tensor of shape (IMG_SIZE, IMG_SIZE, 3)."""
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.uint8)

def load_split(split_df):
    """Decode one split with a parallel tf.data map; return uint8 images and integer labels."""
    paths = tf.constant(split_df["path"].tolist())
    labels = np.array([CLASS_TO_INDEX[label] for label in split_df["label"]], dtype=np.int32)
    images = tf.data.Dataset.from_tensor_slices(paths).map(decode_resize, num_parallel_calls=AUTOTUNE)
    return np.stack(list(images.as_numpy_iterator())), labels

SPLITS = {}
for name in ("train", "validation", "test"):
    split_df = isic2018[isic2018["split"] == name]
    x_path, y_path = CACHE_DIR / f"{name}_x.npy", CACHE_DIR / f"{name}_y.npy"
    if x_path.is_file() and y_path.is_file():
        x, y, source = np.load(x_path), np.load(y_path), "cache"
    else:
        start = time.perf_counter()
        x, y = load_split(split_df)
        np.save(x_path, x)
        np.save(y_path, y)
        source = f"decoded in {time.perf_counter() - start:.1f}s"
    SPLITS[name] = (x, y)
    print(f"{name:10s} {x.shape[0]:5d} images | {x.shape[1]}x{x.shape[2]}x{x.shape[3]} | {x.dtype} | {source}")

def prepare_sample(image, label):
    """Normalise to [0, 1] and one-hot encode the fixed 7-class label."""
    return tf.cast(image, tf.float32) / 255.0, tf.one_hot(label, len(CLASS_NAMES))

x_train, y_train = SPLITS["train"]
x_val, y_val = SPLITS["validation"]
x_test, y_test = SPLITS["test"]

# train_base stays unbatched so 2.5 can apply training-only augmentation before batching.
train_base = tf.data.Dataset.from_tensor_slices((x_train, y_train)).map(prepare_sample, num_parallel_calls=AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(prepare_sample, num_parallel_calls=AUTOTUNE).batch(32).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(prepare_sample, num_parallel_calls=AUTOTUNE).batch(32).prefetch(AUTOTUNE)

train_labels = y_train.copy()          # integer labels kept for class weights in 2.10
class_counts = pd.Series(train_labels).map(CLASS_NAMES.__getitem__).value_counts().reindex(CLASS_NAMES)
print("\ntraining images per class:\n" + class_counts.to_string())
print(f"\ntrain batches per epoch: {math.ceil(len(train_labels) / BATCH_SIZE)}")
for images, labels in val_ds.take(1):
    print("validation batch:", tuple(images.shape), "| labels:", tuple(labels.shape),
          "| one-hot row:", labels[0].numpy().astype(int))

# The datasets now hold the image tensors; release the numpy copies.
del SPLITS, x_train, y_train, x_val, y_val, x_test, y_test


## 2.5–2.6 Augmentation

Required: show two random augmentations of one sample side by side; explain variance without label change. Produces: augmentation figure.

**Augmentation.** Horizontal/vertical flips, small rotations, zoom, translation, contrast and brightness changes are applied only to the training split. They change the pixel statistics of an image - so the model sees extra variance - but never change its label. Validation and test images are never augmented.


In [ ]:
# 2.5 - Training-only augmentation, implemented with Keras preprocessing layers.
AUGMENT = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.08, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomContrast(0.10, seed=SEED),
    tf.keras.layers.RandomBrightness(0.10, value_range=(0.0, 1.0), seed=SEED),
], name="augment")

def augment(image, label):
    image = AUGMENT(image, training=True)
    return tf.clip_by_value(image, 0.0, 1.0), label

train_ds = (train_base
            .shuffle(1000, seed=SEED)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

# 2.6 - One sample and two independent random augmentations of the same sample.
sample_path = sorted((DATA_ROOT / "ISIC 2018").rglob("*.jpg"))[0]
sample = tf.convert_to_tensor(
    np.asarray(Image.open(sample_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0)
panels = [("original", sample),
          ("augmentation 1", AUGMENT(sample, training=True)),
          ("augmentation 2", AUGMENT(sample, training=True))]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, panel) in zip(axes, panels):
    ax.imshow(np.clip(panel.numpy(), 0.0, 1.0))
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("2.5-2.6  same sample, two random augmentations (label unchanged)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()


## 2.7–2.9 Model, summary and feature maps

Required: build the model, print summary, extract feature maps, explain shapes and backpropagation. Produces: summary file, parameter counts, feature-map grid.

**Reading the model.** ResNet-50 first shrinks the spatial size (224 -> 7) while growing the channel depth (3 -> 2048), so the GAP layer sees one 2048-d vector per image; the dense head then maps it to 7 probabilities. Backpropagation starts from the softmax/cross-entropy gradient and applies the chain rule layer by layer, updating the head first and, in stage 2, the residual blocks. Residual shortcuts let the gradient flow around blocks instead of only through them.


In [ ]:
# 2.7 - ResNet-50 backbone (ImageNet weights) + GAP + dropout + softmax head.
base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False      # stage 1 trains the head only; 2.10 unfreezes the last block
model = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="predictions"),
], name="resnet50_isic2018")

summary_lines = []
model.summary(print_fn=summary_lines.append)
summary_text = "\n".join(summary_lines)
(OUT_DIR / "tables" / "model_summary.txt").write_text(summary_text + "\n")
print(summary_text)
trainable_now = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print(f"\ntotal parameters: {model.count_params():,} | trainable in stage 1: {trainable_now:,}")

# 2.8 - Feature maps from one hidden layer for one test image.
for images, labels in test_ds.take(1):
    test_image, test_label = images[0], int(tf.argmax(labels[0]))
feature_layer = base_model.get_layer("conv4_block6_out")
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=feature_layer.output)
feature_maps = feature_extractor(tf.expand_dims(test_image, axis=0), training=False)[0].numpy()
print(f"feature map shape: {feature_maps.shape} | test image class: {CLASS_NAMES[test_label]}")

channels = np.linspace(0, feature_maps.shape[-1] - 1, 16).astype(int)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, channel in zip(axes.ravel(), channels):
    ax.imshow(feature_maps[:, :, channel], cmap="viridis")
    ax.set_title(f"channel {channel}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"2.8  feature maps at conv4_block6_out ({feature_maps.shape[0]}x{feature_maps.shape[1]}x{feature_maps.shape[2]})",
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "feature_maps.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.9 - Why the spatial size shrinks while the channel depth grows (report section 2.9).
print("Input %dx%dx3 -> conv4_block6_out %dx%dx%d: spatial size / %d, channel depth x%.0f"
      % (IMG_SIZE, IMG_SIZE, feature_maps.shape[0], feature_maps.shape[1], feature_maps.shape[2],
         IMG_SIZE // feature_maps.shape[0], feature_maps.shape[2] / 3))


## Part 1.1 support: gradient magnitude per layer, VGG vs ResNet

Required by 1.1: a graph of gradient magnitudes per layer for VGG vs ResNet. Produces: the Part 1 figure and its data table.

In [ ]:
# TODO(1.1): build a VGG and a ResNet with the same input size.
# TODO(1.1): run one forward and one backward pass, record the gradient norm of every layer.
# TODO(1.1): plot gradient magnitude against layer depth for both models and save the figure.

## 2.10–2.13 Training and learning curves

Required: adaptive optimizer, fixed epochs, loss curves with learning rate or gradient norm, overfitting analysis. Produces: training history and curves.

In [ ]:
# TODO(2.10): choose AdamW (adaptive optimizer) and a fixed number of epochs; state the learning-rate schedule.
# TODO(2.10): compute class weights from the training split only.
# TODO(2.11): log per-epoch training loss, validation loss, learning rate and gradient norm.
# TODO(2.12): plot train and validation loss with the learning rate on a second axis.
# TODO(2.13): mark where validation loss starts to rise and write the plateau and overfitting analysis.

## 2.14–2.17 Internal test evaluation

Required: test accuracy and loss, normalised confusion matrix, top-3 confident errors, error analysis. Produces: evaluation figures and tables.

In [ ]:
# TODO(2.14): evaluate the selected weights on the internal test split (accuracy and loss).
# TODO(2.15): plot the row-normalised 7x7 confusion matrix.
# TODO(2.16): show the three wrong predictions with the highest predicted-class probability.
# TODO(2.17): write the error analysis and check it refers to the current final image IDs.

## 2.18–2.20 External dataset, unseen image and calibration

Required: external evaluation, unseen-image probabilities, softmax discussion, calibration. Produces: external table, bar chart, reliability diagram and ECE.


In [ ]:
# TODO(2.18): evaluate on the ISIC 2019 external split (7-class mapping) and report the results table.
# TODO(2.19): pick one fixed unseen ISIC 2019 image, predict it and plot the 7-class bar chart.
# TODO(2.20): explain logits -> softmax -> probabilities, then plot the reliability diagram and report ECE.


## Artifacts, reproducibility and submission packaging

Required: one self-contained notebook and a verified results archive. Produces: weights, metrics JSON, all figures, results ZIP.

In [ ]:
# TODO: save selected weights, metrics JSON and every report figure with fixed file names.
# TODO: store seeds, versions and the split manifest inside the results archive.
# TODO: rerun this notebook top-to-bottom in a fresh local Jupyter session (conda dev) before packaging.
# TODO: export the results ZIP and verify its contents.